# AP3 Einarbeitung: SimPy an einer Kiste (Issue #17)

Lernmodell, **kein Ergebnis für den Verfahrensvergleich**. Es zeigt, wie SimPy funktioniert,
und benutzt dabei schon die echten Bausteine des Repos:

- Parameter aus `config/anlage.yaml` über `matsim.modell.anlage.wert` (AP1, Mediane aus den MES-Daten)
- echte Messwerte je Zeit aus `data/processed/zeitverteilungen.json` (AP1) für Experiment 4
- das Ereignisprotokoll von AP5 (`matsim.kennzahlen.ereignisprotokoll`)

Ablauf je Kiste (Charge): Kommissionierung → Hintransport → Lieferung an den Übergabeplatz →
Belegung der Station → Leerboxrückführung. Das entspricht den Methoden `kommissionierung`,
`hintransport`, `montage` und `leerboxrueckfuehrung` im Gerüst `src/matsim/modell/simulation.py`.

**Kommissionierung** nach der Formel aus AP1 (`projekt/datenmodell-mes.md`, Abschnitt 5.5):
Vorlauf + Teile × Pickzeit + (Quellboxen − 1) × Quellboxwechsel. Der Kommissionierplatz ist dabei
die ganze Zeit belegt (nur ein Auftrag gleichzeitig).

**Vereinfachungen:** kein Bedarfseingang und kein Planungsverfahren (die Chargen stehen fest);
Vorlauf und Quellboxwechsel sind nur Wartezeiten – die Quellbox-Fahrten der Robotinos werden
nicht simuliert, die Fahrzeuge sind in dieser Zeit im Modell frei; Werkstücke stehen sofort an
der Station bereit; Fahrzeuge sind nicht verortet; die Leerboxrückfahrt ist eine Annahme (A-09).

**Starten in VS Code:** einmal `pip install -e ".[dev,notebooks]"` in der `.venv`, oben rechts den
Kernel `.venv` wählen, dann Zelle für Zelle mit `Shift+Enter` oder „Run All“.

In [ ]:
import random
from dataclasses import asdict

import matplotlib.pyplot as plt
import pandas as pd
import simpy

from matsim.daten.zeiten import lade_verteilungen
from matsim.kennzahlen.ereignisprotokoll import Ereignisprotokoll, Ereignistyp, Warteursache
from matsim.modell.anlage import lade_anlage, wert

anlage = lade_anlage()
STATIONEN = list(anlage["stationen"])

PICKZEIT_S = wert(anlage, "kommissionierplatz.pickzeit_je_teil_s")
VORLAUF_S = wert(anlage, "kommissionierplatz.vorlauf_s")
QUELLBOXWECHSEL_S = wert(anlage, "kommissionierplatz.quellboxwechsel_s")
QUELLBOXEN_JE_AUFTRAG = wert(anlage, "kommissionierplatz.quellboxen_je_auftrag")
KOMMISSIONIERPLAETZE = wert(anlage, "kommissionierplatz.gleichzeitige_chargen")
FAHRZEUGE = wert(anlage, "fahrzeuge.anzahl")
FAHRAUFTRAG_S = wert(anlage, "fahrzeuge.fahrauftrag_s")
LEERBOX_RUECK_S = wert(anlage, "fahrzeuge.leerboxruecktransport_s")  # Annahme A-09, keine Messwerte
UEBERGABEPLAETZE = {s: wert(anlage, f"stationen.{s}.uebergabeplaetze") for s in STATIONEN}
BELEGUNG_S = {s: wert(anlage, f"stationen.{s}.belegung_je_werkstueck_s") for s in STATIONEN}
# Teile eines Satzes je Station, z. B. RASS1: [800, 814]
SATZ = {
    s: [pno for pno, e in anlage["stationen"][s]["materialsatz"].items() for _ in range(e["wert"])]
    for s in STATIONEN
}

# Echte Messwerte je Zeit (AP1, Störungen entfernt) – für Experiment 4
VERTEILUNGEN = lade_verteilungen()


def dauer(parameter, rng=None):
    """Ohne Zufall (rng=None) der Wert aus anlage.yaml, sonst ein zufällig gezogener echter Messwert."""
    if rng is None:
        return wert(anlage, parameter)
    return VERTEILUNGEN[parameter].ziehe(rng)


print(f"Pickzeit je Teil {PICKZEIT_S} s, {FAHRZEUGE} Fahrzeuge, Fahrauftrag {FAHRAUFTRAG_S} s")
print(
    f"Vorlauf {VORLAUF_S} s, Quellboxwechsel {QUELLBOXWECHSEL_S} s, {QUELLBOXEN_JE_AUFTRAG} Quellboxen je Auftrag"
)
print(f"Übergabeplätze {UEBERGABEPLAETZE}")
print(f"Belegung je Werkstück {BELEGUNG_S}")
print(f"Satz je Station {SATZ}")
print(
    f"Messwerte für Zufall: {len(VERTEILUNGEN)} Zeiten, z. B. Vorlauf n = {len(VERTEILUNGEN['kommissionierplatz.vorlauf_s'].werte)}"
)

## Die vier Bausteine

1. **`simpy.Environment()`**: die Uhr. `env.now` ist die Simulationszeit in Sekunden. Sie springt
   von Ereignis zu Ereignis.
2. **Prozess**: eine Funktion mit `yield`. `yield env.timeout(23)` heißt „23 s beschäftigt“.
3. **Ressource**: alles Knappe. `anfrage = ressource.request()` stellt sich an, `yield anfrage`
   wartet, bis sie frei ist. Am Ende des `with`-Blocks (oder mit `release`) wird sie wieder frei.
4. **Ereignis**: alles, worauf man mit `yield` warten kann. Eine Anfrage ist auch ein Ereignis.
   `anfrage.triggered` ist schon beim Anstellen `True`, wenn die Ressource frei ist. Damit
   erkennen wir unten, ob überhaupt gewartet wird.

In [ ]:
def warte_auf(env, protokoll, anfrage, ursache, **felder):
    """Wartet auf eine Anfrage; protokolliert Beginn und Ende nur, wenn wirklich gewartet wird."""
    gewartet = not anfrage.triggered
    if gewartet:
        protokoll.schreibe(env.now, Ereignistyp.WARTEN_BEGINN, ursache=ursache, **felder)
    yield anfrage
    if gewartet:
        protokoll.schreibe(env.now, Ereignistyp.WARTEN_ENDE, ursache=ursache, **felder)


def charge(
    env, res, protokoll, charge_id, station, saetze, freigabe=None, geliefert=None, rng=None
):
    """Eine Kiste von der Kommissionierung bis zur Rückkehr der leeren Box."""
    if freigabe is not None:
        yield freigabe  # erst starten, wenn die vorherige Kiste geliefert ist
    ids = {"charge_id": charge_id, "box_id": f"Box_{charge_id}"}
    teile = SATZ[station] * saetze

    # 1) Kommissionierung: Kommissionierplatz belegen, Vorlauf, je Teil ein Pick, Quellboxwechsel
    with res["kommissionierplatz"].request() as platz:
        yield from warte_auf(
            env, protokoll, platz, Warteursache.KOMMISSIONIERUNG, ressource="PICKSORT", **ids
        )
        protokoll.schreibe(
            env.now, Ereignistyp.KOMMISSIONIERUNG_BEGINN, ressource="PICKSORT", **ids
        )
        yield env.timeout(dauer("kommissionierplatz.vorlauf_s", rng))  # Quell- und Zielbox bringen
        for pno in teile:
            yield env.timeout(dauer("kommissionierplatz.pickzeit_je_teil_s", rng))
            protokoll.schreibe(
                env.now, Ereignistyp.TEIL_GEPICKT, ressource="PICKSORT", pno=pno, menge=1, **ids
            )
        # nie mehr Quellboxen als Teile; bei n Quellboxen gibt es n - 1 Wechsel
        quellboxen = min(int(dauer("kommissionierplatz.quellboxen_je_auftrag", rng)), len(teile))
        for _ in range(quellboxen - 1):
            yield env.timeout(dauer("kommissionierplatz.quellboxwechsel_s", rng))
        protokoll.schreibe(env.now, Ereignistyp.KOMMISSIONIERUNG_ENDE, ressource="PICKSORT", **ids)

    # 2) Hintransport und Lieferung: Fahrzeug bleibt belegt, bis ein Übergabeplatz frei ist
    with res["fahrzeuge"].request() as fahrzeug:
        yield from warte_auf(
            env, protokoll, fahrzeug, Warteursache.TRANSPORT, ressource="fahrzeug", **ids
        )
        protokoll.schreibe(env.now, Ereignistyp.TRANSPORT_BEGINN, ressource="fahrzeug", **ids)
        yield env.timeout(dauer("fahrzeuge.fahrauftrag_s", rng))
        protokoll.schreibe(env.now, Ereignistyp.TRANSPORT_ENDE, ressource=station, **ids)
        uebergabe = res["uebergabeplaetze"][station].request()  # ohne with: Freigabe erst in 4)
        yield from warte_auf(
            env, protokoll, uebergabe, Warteursache.STATIONSKAPAZITAET, ressource=station, **ids
        )
        protokoll.schreibe(env.now, Ereignistyp.LIEFERUNG, ressource=station, **ids)
    if geliefert is not None:
        geliefert.succeed()  # gibt die nächste Kiste frei (Experiment 3)

    # 3) Montage, vereinfacht: die Station ist je Satz so lange belegt wie ein Werkstück
    protokoll.schreibe(env.now, Ereignistyp.MONTAGE_BEGINN, ressource=station, **ids)
    for _ in range(saetze):
        yield env.timeout(dauer(f"stationen.{station}.belegung_je_werkstueck_s", rng))
    protokoll.schreibe(env.now, Ereignistyp.MONTAGE_ENDE, ressource=station, **ids)

    # 4) Leerboxrückführung: Fahrzeug holen, Übergabeplatz freigeben, zurück zum Lager
    with res["fahrzeuge"].request() as fahrzeug:
        yield from warte_auf(
            env, protokoll, fahrzeug, Warteursache.TRANSPORT, ressource="fahrzeug", **ids
        )
        res["uebergabeplaetze"][station].release(uebergabe)
        protokoll.schreibe(env.now, Ereignistyp.LEERBOX_RUECK_BEGINN, ressource=station, **ids)
        yield env.timeout(LEERBOX_RUECK_S)  # Annahme A-09, auch mit Zufall fest
        protokoll.schreibe(env.now, Ereignistyp.LEERBOX_RUECK_ENDE, ressource="lager", **ids)


def laufe(chargen, fahrzeuge=FAHRZEUGE, uebergabeplaetze=None, nacheinander=False, rng=None):
    """Baut das Modell auf, startet je Charge einen Prozess und lässt die Uhr laufen.

    rng=None: alle Zeiten aus anlage.yaml (jeder Lauf gleich).
    rng=random.Random(seed): Zeiten zufällig aus den echten Messwerten (jeder Seed ein anderer Lauf).
    """
    env = simpy.Environment()
    protokoll = Ereignisprotokoll()
    plaetze = (
        UEBERGABEPLAETZE if uebergabeplaetze is None else dict.fromkeys(STATIONEN, uebergabeplaetze)
    )
    res = {
        "kommissionierplatz": simpy.Resource(env, capacity=KOMMISSIONIERPLAETZE),
        "fahrzeuge": simpy.Resource(env, capacity=fahrzeuge),
        "uebergabeplaetze": {s: simpy.Resource(env, capacity=n) for s, n in plaetze.items()},
    }
    vorgaenger = None
    for charge_id, station, saetze in chargen:
        geliefert = env.event() if nacheinander else None
        env.process(
            charge(env, res, protokoll, charge_id, station, saetze, vorgaenger, geliefert, rng)
        )
        vorgaenger = geliefert
    env.run()  # läuft, bis keine Ereignisse mehr anstehen
    protokoll.schreibe(env.now, Ereignistyp.LAUF_ENDE)
    return protokoll


def mmss(sekunden):
    return f"{int(sekunden // 60):02d}:{int(sekunden % 60):02d}"


def als_tabelle(protokoll, picks=False):
    """Protokoll als Tabelle; einzelne Picks nur auf Wunsch."""
    df = pd.DataFrame([asdict(e) for e in protokoll.ereignisse])
    if not picks:
        df = df[df["ereignis"] != Ereignistyp.TEIL_GEPICKT]
    df = df.assign(zeit=df["zeit_s"].map(mmss), ereignis=df["ereignis"].astype(str))
    spalten = ["zeit", "ereignis", "charge_id", "ressource", "ursache", "pno"]
    return df[spalten].replace({0: "", "": ""}).reset_index(drop=True)


def kennzahlen(protokoll):
    """Laufende und Wartezeit je Ursache in Sekunden (Summe Ende minus Summe Beginn)."""
    df = pd.DataFrame([asdict(e) for e in protokoll.ereignisse])
    warten = df[df["ereignis"].isin([Ereignistyp.WARTEN_BEGINN, Ereignistyp.WARTEN_ENDE])]
    vorzeichen = warten["ereignis"].map({Ereignistyp.WARTEN_BEGINN: -1, Ereignistyp.WARTEN_ENDE: 1})
    je_ursache = (warten["zeit_s"] * vorzeichen).groupby(warten["ursache"]).sum()
    return {
        "ende": mmss(df["zeit_s"].max()),
        **{f"warten_{u}_s": int(t) for u, t in je_ursache.items()},
    }

## Basislauf: drei Kisten wie am 09.11.2026

Je Station eine Box mit drei Sätzen (9176/1, 9178/1, 9178/2), alle Kisten zum Zeitpunkt 0.

In [ ]:
CHARGEN_0911 = [("K9177", "RASS1", 3), ("K9179", "RASS2", 3), ("K9180", "RASS3", 3)]

basis = laufe(CHARGEN_0911)
print(kennzahlen(basis))
als_tabelle(basis)

## Experiment 1: Wie viele Fahrzeuge braucht es?

Ändere die Liste und führe die Zelle erneut aus. Wo verschiebt sich die Wartezeit hin?
Mit Vorlauf und Quellboxwechsel endet der Lauf bei 1, 2 und 3 Fahrzeugen zur selben Zeit; mit einem
Fahrzeug wartet eine Kiste nur kurz. Warum – und was ändert sich, wenn die Quellbox-Fahrten (die heute
im Vorlauf stecken) später als eigene Robotino-Fahrten simuliert werden?

In [ ]:
pd.DataFrame(
    {f"{n} Fahrzeug(e)": kennzahlen(laufe(CHARGEN_0911, fahrzeuge=n)) for n in [1, 2, 3]}
).fillna(0)

## Experiment 2: Ein Übergabeplatz, zwei Boxen für RASS3

Die zweite Box kommt an, während die erste noch an der Station steht. Das Fahrzeug wartet
beladen vor der Station (Ursache `stationskapazitaet`) und fehlt in der Zeit für andere Fahrten.
Ohne Vorlauf und Quellboxwechsel waren das 320 s – unten taucht die Wartezeit gar nicht mehr auf.
Warum? (Tipp: Vergleiche, wann K2 fertig kommissioniert ist und wann K1 an der Station fertig ist.)

In [ ]:
voll = laufe([("K1", "RASS3", 8), ("K2", "RASS3", 8)], uebergabeplaetze=1)
print(kennzahlen(voll))
als_tabelle(voll)

## Experiment 3: Stationen nacheinander wie im echten Lauf

Am 09.11.2026 entstand der Kommissionierauftrag für die nächste Station erst, als die Box der
vorherigen abgeliefert war. `nacheinander=True` bildet das über ein eigenes Ereignis ab
(`env.event()` und `geliefert.succeed()` in `charge`).

Echt lagen zwischen dem Anlegen für RASS1 (15:05:15) und der Box an RASS3 (15:45:09) **39:54 min**.
Ohne Vorlauf und Quellboxwechsel kam das Modell auf 10:33, mit beiden kommt es auf 26:51 (unten).
Der Rest: Am 09.11. brauchte jeder Auftrag 3–4 Quellboxen (Median: 2), die Wechselpausen waren länger
als der Median, und die Robotinos waren nicht immer sofort frei. Ist der 09.11. also „falsch“
modelliert – oder nur ein langsamer Tag? Das klärt Experiment 4.

In [ ]:
nacheinander = laufe(CHARGEN_0911, nacheinander=True)
lieferungen = als_tabelle(nacheinander).query("ereignis == 'lieferung'")
lieferungen

## Experiment 4: Zufall – wie stark streut die Lieferzeit?

Bisher nimmt das Modell für jede Zeit **einen** Wert (Median aus `anlage.yaml`), deshalb kommt jedes
Mal dasselbe heraus. In echt streuen die Zeiten: Der Vorlauf lag bei den 229 Aufträgen zwischen
gut 3 und über 13 Minuten. AP1 hat dafür alle echten Messwerte je Zeit abgelegt
(`data/processed/zeitverteilungen.json`, Störungen entfernt).

Mit `rng=random.Random(seed)` zieht `dauer()` für **jeden** Vorlauf, jeden Pick, jede Fahrt usw. einen
zufälligen echten Messwert (Ziehen mit Zurücklegen). Ein Durchlauf mit einem Seed heißt **Replikation**.
Derselbe Seed liefert immer denselben Lauf – so bleibt das Ergebnis nachvollziehbar.

Wir wiederholen Experiment 3 `REPLIKATIONEN`-mal und schauen, wann die Box an RASS3 ankommt.
Das **10-%-Quantil** heißt: 10 % der Läufe sind schneller; das **90-%-Quantil**: 10 % sind langsamer.

In [ ]:
REPLIKATIONEN = 1000
ECHT_S = 39 * 60 + 54  # Box an RASS3 am 09.11.2026, ab Anlage des ersten Kommissionierauftrags


def lieferung_rass3_s(protokoll):
    """Zeitpunkt, zu dem die Box für RASS3 (Charge K9180) am Übergabeplatz steht."""
    return next(
        e.zeit_s
        for e in protokoll.ereignisse
        if e.ereignis == Ereignistyp.LIEFERUNG and e.charge_id == "K9180"
    )


ohne_zufall = lieferung_rass3_s(laufe(CHARGEN_0911, nacheinander=True))
mit_zufall = pd.Series(
    [
        lieferung_rass3_s(laufe(CHARGEN_0911, nacheinander=True, rng=random.Random(seed)))
        for seed in range(REPLIKATIONEN)
    ]
)

pd.Series(
    {
        "ohne Zufall (Werte aus anlage.yaml)": mmss(ohne_zufall),
        "mit Zufall: 10-%-Quantil": mmss(mit_zufall.quantile(0.10)),
        "mit Zufall: Median": mmss(mit_zufall.median()),
        "mit Zufall: 90-%-Quantil": mmss(mit_zufall.quantile(0.90)),
        "echt am 09.11.2026": mmss(ECHT_S),
        "Anteil der Läufe, die so langsam sind wie echt": f"{(mit_zufall >= ECHT_S).mean():.0%}",
    },
    name="Box an RASS3",
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(mit_zufall / 60, bins=40, color="#2a78d6", edgecolor="white", linewidth=1)
for x, text, stil in [
    (ohne_zufall / 60, "ohne Zufall", "--"),
    (ECHT_S / 60, "echt 09.11.", ":"),
]:
    ax.axvline(x, color="#52514e", linewidth=1.5, linestyle=stil)
    ax.annotate(
        text,
        (x, 1),
        xycoords=("data", "axes fraction"),
        xytext=(4, -12),
        textcoords="offset points",
        color="#52514e",
        fontsize=9,
    )
ax.set(
    title=f"Experiment 4: Box an RASS3 in {REPLIKATIONEN} Läufen mit echten Zufallszeiten",
    xlabel="Box an RASS3 nach … Minuten",
    ylabel="Anzahl Läufe",
)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.show()

**Zum Ausprobieren**

1. Warum liegt der Median *mit* Zufall über dem Wert *ohne* Zufall, obwohl beide aus denselben Daten kommen?
   (Tipp: Schau dir an, wie lang ein Vorlauf höchstens und wie kurz er mindestens sein kann.)
2. Setze `REPLIKATIONEN = 10`, dann `100`. Ab wann ändern sich die Quantile kaum noch?
3. Ändere in `lieferung_rass3_s` die Charge auf `"K9177"` (RASS1). Wie breit streut eine einzelne Station?

## Zum Schluss: Welche SimPy-Bausteine wofür?

| Im Modell | SimPy-Baustein | Warum dieser? |
|---|---|---|
| Uhr / Simulationszeit | `simpy.Environment()`, aktuelle Zeit `env.now` | Die Uhr springt von Ereignis zu Ereignis, statt Sekunde für Sekunde zu ticken. `env.run()` läuft, bis nichts mehr ansteht. |
| Eine Kiste auf ihrem Weg | **Prozess**: Funktion `charge()` mit `yield`, gestartet mit `env.process(...)` | Die Kiste hat einen eigenen Ablauf (kommissionieren → fahren → liefern → montieren → zurück), in dem sie immer wieder wartet. Jede Kiste ist ein eigener Prozess und läuft parallel zu den anderen. |
| Kommissionierplatz | **Ressource** `simpy.Resource(env, capacity=1)`, belegt mit `with ... .request()` | Nur ein Auftrag gleichzeitig. Wer kommt, während der Platz belegt ist, stellt sich an (Wartezeit `kommissionierung`). Am Ende des `with`-Blocks wird der Platz automatisch frei. |
| Fahrzeuge | **Ressource** `simpy.Resource(env, capacity=2)` | Es gibt nur zwei Robotinos. Ist keiner frei, wartet die Kiste (`transport`). Ein Fahrzeug bleibt belegt, bis die Box am Übergabeplatz steht. |
| Übergabeplätze je Station | je Station eine **Ressource** mit `capacity=2`; `request()` **ohne** `with`, `release()` erst bei der Leerboxrückführung | Die Box belegt den Platz über mehrere Abschnitte hinweg (Lieferung, Montage, bis die Leerbox abgeholt wird). Das passt nicht in einen einzigen `with`-Block, deshalb wird von Hand freigegeben. |
| „Zeit vergeht“ (Pick, Fahrt, Montage) | `yield env.timeout(sekunden)` | Der Prozess ist für eine Dauer beschäftigt. In dieser Zeit laufen die anderen Prozesse weiter. |
| „Box ist abgeliefert“ (Experiment 3) | **Ereignis** `env.event()`, ausgelöst mit `geliefert.succeed()`, abgewartet mit `yield freigabe` | Ein Signal zwischen zwei Prozessen: Die nächste Kiste startet erst, wenn die vorige geliefert ist. Kein Platz und keine Dauer, sondern ein Zeitpunkt, auf den jemand wartet. |
| Wartet die Kiste überhaupt? | `anfrage.triggered` (in `warte_auf`) | Ist die Anfrage beim Anstellen schon erfüllt, war die Ressource frei. Nur sonst wird „Warten“ protokolliert. |
| Zufallszeiten (Experiment 4) | kein SimPy, sondern Python: `random.Random(seed)` und `dauer(parameter, rng)` mit den AP1-Messwerten | SimPy wartet nur so lange wie angegeben – woher die Dauer kommt, entscheiden wir. Ein eigener Zufallsgenerator je Replikation sorgt dafür, dass derselbe Seed denselben Lauf ergibt. |

**Faustregel:** Wer einen eigenen Ablauf hat, ist ein *Prozess*. Was knapp ist und worauf man warten muss, ist eine *Ressource*. Ein Zeitpunkt, auf den jemand wartet, ist ein *Ereignis*. Eine Dauer ist ein *Timeout*.

**Was davon braucht der Simulationskern (`src/matsim/modell/simulation.py`) genauso?**
Alles oben: die Uhr, je Kiste ein Prozess, Ressourcen für Kommissionierplatz, Fahrzeuge und Übergabeplätze,
Timeouts für die Dauern, `dauer()` mit Zufallsgenerator für das Szenario „Unsicherheit“ und das Ereignisprotokoll.

**Was fehlt dort noch?**
- Quellbox-Fahrten als eigene Robotino-Aufträge (Variante c statt Vorlauf als Wartezeit)
- Werkstücke als eigene Prozesse mit Umlauf und Montage nur bei vollständigem Satz (#28)
- Übergabe der Leerbox an die Rückführung ins Kistenlager
- Bedarfseingang und Planungsaufrufe mit Wartefristen als Ereignisse (#27)
- eine Regel, was bei gleichzeitigen Ereignissen zuerst passiert